# D-MeZO Week-1 final sprint — run-all notebook

**Что делает этот notebook:** последовательно прогоняет все оставшиеся эксперименты Week-1.

| section | runs | wall-clock | compute units |
|---|---|---|---|
| 0. Setup | — | <1 min | 0 |
| 1. **R1d** Nesterov-MeZO с β-decay | 1 | ~37 min | ~3-5 |
| 2. Phase 3a: centralized Qwen3.5 baseline | 1 | ~37 min | ~3-5 |
| 3. Phase 3b: Day 5 grid retrofit seed=42 (accuracy retrofit) | 4 | ~2.5h | ~6-8 |
| 4. Phase 3c: Day 5 grid seed=43 (variance) | 4 | ~2.5h | ~6-8 |
| 5. Final analysis — aggregated tables + diagrams | — | ~1 min | 0 |
| **TOTAL** | **10** | **~6h** | **~20-25 units** |

**Перед запуском:** репо `Siesher/dmezo` приватный, нужен PAT.
Два варианта:
- (Recommended) Colab Secrets: левая панель -> ключ -> name=`GH_PAT`, value=твой PAT.
- (Fallback) Если Secret не создан, cell 3 спросит PAT через getpass (input скрыт).
Сгенерировать PAT: https://github.com/settings/tokens (нужен scope `repo`).

**Как запустить:**

1. Откройте этот notebook в Colab Pro+ (нужно A100 / L40 / H100 GPU — для Qwen3.5-4B в bf16).
2. Runtime → Run all (или Ctrl+F9). Cells будут выполняться по очереди.
3. **Не закрывайте вкладку** — Colab убивает sessions при закрытии браузера.
4. Через ~6 часов проверьте результат в последнем cell.

**Что делать если session умерла посередине:**
- Все output_dirs под Google Drive (`/content/drive/MyDrive/dmezo_runs/`) — данные сохраняются.
- MLflow runs тоже под Drive — agregate cell найдёт их при повторном запуске.
- Можно перезапустить с того config-а, который не успел (закомментируй уже сделанные в loop'ах).

**Что мы ожидаем:**

| run | expected final eval | vs control 0.1381 |
|---|---|---|
| R1d (β-decay) | 0.10-0.18 (надежда: <0.13) | should beat |
| centralized baseline | 0.08-0.12 | reference |
| Day 5 retrofit (seed=42) | matches Day 5 originals | — |
| Day 5 seed=43 | within ~10% of seed=42 | error bar |


## 0. Setup (Drive mount + repo + deps + GPU check)

In [ ]:
# Mount Google Drive — runs persist across session deaths.
from google.colab import drive
drive.mount('/content/drive')

import os
RUNS_DIR = '/content/drive/MyDrive/dmezo_runs'
os.makedirs(RUNS_DIR, exist_ok=True)
print(f'RUNS_DIR = {RUNS_DIR}')


In [ ]:
# Clone repo (private - uses a Personal Access Token from Colab Secrets,
# with a getpass fallback if no Secret is configured).
import os
import subprocess

GH_REPO = 'Siesher/dmezo'

if not os.path.exists('/content/dmezo/.git'):
    # Resolve a PAT for the private clone. Two paths, in order of preference:
    #   1. Colab Secrets - add a Secret named GH_PAT (left sidebar, key icon)
    #   2. getpass prompt - fallback that won't echo the token into the cell output
    PAT = None
    try:
        from google.colab import userdata  # type: ignore
        PAT = userdata.get('GH_PAT')
    except Exception:
        PAT = None
    if not PAT:
        import getpass
        PAT = getpass.getpass('GitHub PAT (input hidden): ')

    # Use subprocess (not !git clone) so the PAT-bearing URL is not echoed.
    subprocess.run(
        ['git', 'clone', f'https://{PAT}@github.com/{GH_REPO}.git', '/content/dmezo'],
        check=True,
    )
    del PAT  # remove token from memory
else:
    # Already cloned - just refresh.
    subprocess.run(['git', '-C', '/content/dmezo', 'pull'], check=True)

# Show last three commits to confirm we have latest main.
subprocess.run(['git', '-C', '/content/dmezo', 'log', '--oneline', '-3'], check=True)


In [ ]:
# Install package + light deps. transformers / torch already on Colab.
!pip install -q --upgrade pip
!cd /content/dmezo && pip install -q -e .


In [ ]:
# GPU sanity check — must have a CUDA GPU with bf16 support.
import torch
assert torch.cuda.is_available(), 'No CUDA GPU available — change runtime to GPU.'
gpu = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} ({mem_gb:.1f} GB)')
print(f'CUDA: {torch.version.cuda}, torch: {torch.__version__}')
# Memory needed: 4x Qwen3.5-4B bf16 + activations ~= 55-60 GB peak.
if mem_gb < 40:
    print('\nWARNING: GPU has less than 40 GB. You may OOM on 4-client runs.')
    print('Consider switching to A100/L40/H100 runtime.')


In [ ]:
# Set common environment variables for all runs.
import os
MLFLOW_URI = f'file:{RUNS_DIR}/mlruns'
os.chdir('/content/dmezo')   # all subsequent !python expects cwd = repo root
print(f'MLFLOW_URI = {MLFLOW_URI}')
print(f'cwd        = {os.getcwd()}')


## 1. R1d — Nesterov-MeZO с β-decay schedule

**Главный новый эксперимент Day 8.**

Настройка: ring(4) + Dirichlet(α=0.5) — самая сложная ячейка Day 5. β линейно затухает с 0.9 (t=0) до 0.0 (t=999). ρ-clipping = 50 (от R1b). Все остальные параметры идентичны R1b и control.

**Гипотеза:** R1b (β=0.9 const) дал 3× early acceleration (best 0.119@R300), но дрейфил late (final 0.225). R1d должен сохранить early phase (β≈0.9 в начале) и убрать late drift (β→0 к концу).

| исход | вывод |
|---|---|
| final < 0.13 | best-of-both-worlds — strong positive |
| 0.13-0.17 | drift fixed, нет net acceleration |
| 0.17-0.22 | schedule слишком агрессивный |
| > 0.22 | unexpected; нужен другой shape |

In [ ]:
# R1d run (~37 min)
SHORT = 'dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2_nest_decay'
CFG = f'configs/{SHORT}.yaml'
OUT_R1D = f'{RUNS_DIR}/{SHORT}'
!python scripts/03_dmezo_federated.py \
    --config {CFG} \
    --output-dir {OUT_R1D} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name {SHORT}
print(f'\nR1d done. Logs in {OUT_R1D}/console.log')


## 2. Phase 3a — Centralized Qwen3.5 baseline (Day 7 E2)

Clean apples-to-apples reference для Day 5 federated grid:
- та же модель Qwen3.5-4B-Base
- тот же train pool (2000 examples = 4×500 за все клиенты)
- 1000 шагов (= num_rounds)
- те же lr/eps/seed=42

Закрывает вопрос "а федерация вообще похожа на centralized?".

**Expected:** drop ~96% (~0.10-0.12 final eval, как Day 2 централизованный Qwen3.5).


In [ ]:
# Phase 3a run (~37 min)
SHORT_C = 'centralized_qwen3_5_4b_base_sst2_2000'
CFG_C = f'configs/{SHORT_C}.yaml'
OUT_C = f'{RUNS_DIR}/{SHORT_C}'
!python scripts/01_sanity_check_mezo.py \
    --config {CFG_C} \
    --output-dir {OUT_C} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name {SHORT_C}
print(f'\nCentralized baseline done. Logs in {OUT_C}/console.log')


## 3. Phase 3b — Day 5 grid retrofit seed=42 (accuracy)

Те же 4 конфига что Day 5, тот же seed=42 — но теперь скрипт логирует **accuracy** на каждом eval-шаге (раньше его не было). Trajectory eval_loss должна совпадать bit-exact с оригинальными Day 5 runs; добавится только accuracy column.

~2.5 ч (4 runs × ~37 min).

In [ ]:
# Phase 3b - 4 runs x ~37 min = ~2.5 h
DAY5_CONFIGS = [
    'configs/dmezo_4c_complete_iid_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_complete_dir05_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_ring_iid_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2.yaml',
]

for cfg in DAY5_CONFIGS:
    short = cfg.split('/')[-1].replace('.yaml', '') + '_s42'
    out = f'{RUNS_DIR}/{short}'
    print('\n' + '=' * 70)
    print(f'  {short}')
    print('=' * 70)
    !python scripts/03_dmezo_federated.py \
        --config {cfg} \
        --output-dir {out} \
        --mlflow-uri {MLFLOW_URI} \
        --run-name {short} \
        --seed 42


## 4. Phase 3c — Day 5 grid seed=43 (variance estimate)

Те же 4 конфига, но seed=43. Совместно с seed=42 runs даст mean ± range за 2 seeds — необходимо для error bars в paper.

~2.5 ч.

In [ ]:
# Phase 3c - 4 runs x ~37 min = ~2.5 h
for cfg in DAY5_CONFIGS:
    short = cfg.split('/')[-1].replace('.yaml', '') + '_s43'
    out = f'{RUNS_DIR}/{short}'
    print('\n' + '=' * 70)
    print(f'  {short}')
    print('=' * 70)
    !python scripts/03_dmezo_federated.py \
        --config {cfg} \
        --output-dir {out} \
        --mlflow-uri {MLFLOW_URI} \
        --run-name {short} \
        --seed 43


## 5. Final analysis — aggregated tables

Сводный отчёт по всему Week 1, готовый для one-pager.


In [ ]:
# Aggregate MLflow runs from all experiments - final summary.
import json
import os
import mlflow
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_URI)

DAY4_CONTROL = 0.1793   # Qwen3-4B 2c complete IID (Day 4 baseline)
DAY5_CONTROL_RING_DIR = 0.1381  # Day 5 cell 4 - worst non-Nesterov


def search(exp_name):
    e = mlflow.get_experiment_by_name(exp_name)
    if e is None:
        return pd.DataFrame()
    return mlflow.search_runs(experiment_ids=[e.experiment_id])


# ---- 1. Day 5 grid: mean +/- range over seeds {42, 43}
print('=' * 72)
print('  TABLE 1 - Day 5 grid mean +/- range (seeds 42, 43)')
print('=' * 72)
day5 = search('dmezo_federated_day5')
if not day5.empty:
    mask = day5['tags.mlflow.runName'].str.contains('_s4[23]$', regex=True, na=False)
    grid = day5[mask].copy()
    grid['config'] = grid['tags.mlflow.runName'].str.replace('_s\\d+$', '', regex=True)

    agg = grid.groupby('config').agg(
        final_loss_mean=('metrics.final_eval_loss', 'mean'),
        final_loss_std=('metrics.final_eval_loss', 'std'),
        final_acc_mean=('metrics.final_eval_acc', 'mean'),
        final_acc_std=('metrics.final_eval_acc', 'std'),
        n_seeds=('metrics.final_eval_loss', 'count'),
    ).reset_index().sort_values('config')

    def fmt(x):
        return f'{x:.4f}' if isinstance(x, float) else str(x)
    print(agg.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
else:
    print('No Day 5 multi-seed runs found (run Phase 3b/3c first).')

# ---- 2. Centralized baseline
print()
print('=' * 72)
print('  TABLE 2 - Centralized Qwen3.5 baseline (Phase 3a)')
print('=' * 72)
centr = search('dmezo_centralized_baseline_day7')
if not centr.empty:
    last = centr.iloc[0]
    fl = last.get('metrics.final_eval_loss', float('nan'))
    fa = last.get('metrics.final_eval_acc', float('nan'))
    dr = last.get('metrics.eval_loss_drop_pct', float('nan'))
    print(f'  final_eval_loss = {fl:.4f}')
    print(f'  final_eval_acc  = {fa:.4f}')
    print(f'  drop            = {dr:.1f}%')
else:
    print('No centralized baseline found.')

# ---- 3. Nesterov phase diagram
print()
print('=' * 72)
print('  TABLE 3 - Nesterov-MeZO phase diagram (Day 6 + 6b + 8)')
print('=' * 72)
phase_rows = []
for exp_name in ['dmezo_federated_day6', 'dmezo_federated_day8_rescue']:
    runs = search(exp_name)
    if runs.empty:
        continue
    for _, r in runs.iterrows():
        phase_rows.append({
            'run': r.get('tags.mlflow.runName', '?'),
            'beta': r.get('params.nesterov.beta', '?'),
            'rho_clip': r.get('params.mezo.rho_clip', 'none'),
            'beta_end': r.get('params.nesterov.beta_end', '-'),
            'final_loss': r.get('metrics.final_eval_loss', float('nan')),
            'final_acc': r.get('metrics.final_eval_acc', float('nan')),
        })
phase = pd.DataFrame(phase_rows).sort_values('run')
if not phase.empty:
    print(phase.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
else:
    print('No Nesterov ablation runs found.')


# ---- 4. R1d-specific deep dive (per-round trajectory)
print()
print('=' * 72)
print('  TABLE 4 - R1d vs R1b vs control (per-round eval trajectory)')
print('=' * 72)

def load_trajectory(short):
    # Load eval_loss + eval_acc at each round from log.jsonl.
    path = f'{RUNS_DIR}/{short}/log.jsonl'
    if not os.path.exists(path):
        return None
    rows = []
    with open(path) as f:
        for line in f:
            d = json.loads(line)
            if 'eval_loss' in d:
                rows.append({
                    'round': d.get('round', 0),
                    'eval_loss': d['eval_loss'],
                    'eval_acc': d.get('eval_acc', None),
                })
    return pd.DataFrame(rows) if rows else None


trajs = {}
for short, label in [
    ('dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2_nest_decay', 'R1d (b-decay)'),
    ('dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2_nest09_clip50', 'R1b (b=0.9 const)'),
    ('dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2_s42', 'control (no Nest)'),
]:
    t = load_trajectory(short)
    if t is not None:
        trajs[label] = t

if trajs:
    pivot = pd.DataFrame({label: t.set_index('round')['eval_loss'] for label, t in trajs.items()})
    print('eval_loss per round:')
    print(pivot.to_string(float_format=lambda x: f'{x:.4f}'))
else:
    print('No R1d trajectory yet (R1d run not complete).')

print()
print('=' * 72)
print('  DONE - copy these tables into docs/09-one-pager.md')
print('=' * 72)
